# Critical-Reasoning SLM — Showcase

A **Qwen3.5-4B + LoRA** fine-tune (`v7`) that reads a self-contained logic problem, reasons
briefly from the premises, and **commits to exactly one credited answer** — including recognizing
**under-determination** (the neutral "does-not-follow" class that frontier models handle poorly).

## Results — held-out clean eval (n=60/family)

Neutral-recall in parens. **MVR** = LGMT label-flip rate under a logic-preserving edit (lower = better).

| Model | **mean** | arct | folio (neu) | logicnli (neu) | logiqa | lsat_lr | proverqa (neu) | MVR/HDR |
|---|--:|--:|--:|--:|--:|--:|--:|--:|
| base 4B (untuned) | 63.9 | 75.0 | 61.7 (50.0) | 48.3 (26.7) | 66.7 | 81.7 | 50.0 (63.3) | 20/20 |
| **v7 (this model)** | **85.3** | 91.7 | 71.7 (80.0) | 83.3 (90.0) | 80.0 | 91.7 | 93.3 (96.7) | **0/0** |
| Sonnet 4.6 | 79.6 | 88.3 | 80.0 (76.7) | 60.0 (26.7) | 85.0 | 94.9 | 69.5 (73.3) | 25/10 |
| Opus 4.8 | 87.2 | 90.0 | 78.3 (80.0) | 93.3 (90.0) | 83.3 | 98.3 | 80.0 (90.0) | 10/10 |

**Reads:** +21.4 over the untuned 4B base; **beats Sonnet 4.6 outright** and sits between Sonnet
and Opus (within 1.9); **most robust of all four (MVR 0)** — beating both frontier models — and
**beats both** on ProverQA. Residual gap to Opus is logicnli / lsat_lr / folio (4B depth ceiling).

Run the cells below to load the deployed model and try it yourself.

In [ ]:
# Option A (deployed): load from HuggingFace — no Drive needed.
#   BASE = "Qwen/Qwen3.5-4B"; ADAPTER = "YOUR-USERNAME/critical-reasoning-slm-v7"
#
# Option B (from your Drive training run):
from google.colab import drive
drive.mount("/content/drive")

BASE    = "Qwen/Qwen3.5-4B"
ADAPTER = "/content/drive/MyDrive/SLM/runs/colab/adapter"   # or your HF repo id


In [ ]:
!pip -q install "transformers>=4.51" "peft>=0.11" "accelerate>=0.30" safetensors

In [ ]:
# Load base + v7 LoRA adapter (bf16, GPU)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
_base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(_base, ADAPTER).eval()
print("loaded v7 adapter on", BASE, "|", next(model.parameters()).device)

In [ ]:
# Inference helpers — same house prompt + greedy, enable_thinking=False (as trained)
@torch.no_grad()
def _gen(prompt, max_new_tokens=512):
    ids = tok.apply_chat_template(
        [{"role": "user", "content": prompt}], add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def solve_entailment(premises, conclusion):
    """Entailment core (FRQ): commits to True / False / Uncertain."""
    stim = "Premises:\n" + "\n".join(f"- {p}" for p in premises) + f"\n\nConclusion: {conclusion}"
    prompt = ("You are an expert in logic and argument analysis. Answer the following open-ended "
              "question about the argument. Reason carefully and commit to the single best answer - "
              "do not hedge by listing many possibilities. Be specific and concise.\n\n"
              f"{stim}\n\nQuestion: Based only on the premises, is the conclusion True, False, or "
              "Uncertain (i.e. it does not deductively follow either way)?\n\nExplain your reasoning "
              "in a few sentences, then clearly state your final conclusion as exactly one of the "
              "labels named in the question.")
    print(_gen(prompt))

def solve_mcq(passage, question, options):
    """Argument satellite (MCQ): commits to one option letter."""
    opts = "\n".join(f"({chr(65 + i)}) {o}" for i, o in enumerate(options))
    prompt = ("You are an expert in logic and argument analysis. Read the passage and the options, "
              "then choose the SINGLE best option.\n\n"
              f"{passage}\n\nQuestion: {question}\n\nOptions:\n{opts}\n\n"
              "Reason through it in a few sentences, then clearly commit to your final answer, "
              "naming the single correct option by its letter.")
    print(_gen(prompt))

## Demo 1 — recognizing under-determination (the neutral class)

The flagship weakness the model was built to fix: knowing when a conclusion **doesn't follow**.

In [ ]:
solve_entailment(
    ["All dogs are mammals.",
     "Some mammals are pets.",
     "Rex is a dog."],
    "Rex is a pet.")
# Expected: Uncertain — dogs are mammals, but only *some* mammals are pets;
# nothing in the premises pins Rex as one of them.

## Demo 2 — multi-step entailment (a definite answer exists)

In [ ]:
solve_entailment(
    ["Every student who studies hard passes the exam.",
     "Everyone who passes the exam graduates.",
     "Mia is a student who studies hard."],
    "Mia graduates.")
# Expected: True — studies hard -> passes -> graduates.

## Demo 3 — argument analysis (MCQ)

In [ ]:
solve_mcq(
    "The new cafe must be excellent: it is always crowded, and people would not keep "
    "going to a cafe that was not excellent.",
    "The reasoning above is most vulnerable to criticism on the grounds that it",
    ["takes for granted what it sets out to establish — that the crowds reflect quality "
     "rather than, say, low prices or a convenient location",
     "relies on a survey with too small a sample",
     "confuses the cause of the crowding with its effect",
     "draws a conclusion about all cafes from one cafe"])
# Expected: (A) — circular; the crowd could reflect price/location, not quality.

## Demo 4 — robustness (consistency under an irrelevant premise)

The same problem as Demo 1, but with an unrelated fact bolted on. A reliable reasoner ignores it —
v7 scores **MVR 0** on the held-out consistency probe.

In [ ]:
solve_entailment(
    ["All dogs are mammals.",
     "Some mammals are pets.",
     "Rex is a dog.",
     "The integer 7 is a prime number."],   # <- irrelevant
    "Rex is a pet.")
# Expected: still Uncertain — the extra fact changes nothing.

## Try it yourself\nEdit the premises/conclusion (or use `solve_mcq`) and re-run.

In [ ]:
solve_entailment(
    ["<premise 1>",
     "<premise 2>"],
    "<your conclusion>")

# MCQ form:
# solve_mcq("<passage>", "<question>", ["<option A>", "<option B>", "<option C>", "<option D>"])